# MHT-CET College Predictor — Ensemble ML Model (Colab)

Trains and evaluates the real, genuine ensemble model for CampusCompass_AI /
Mission Admission A2Z (CETWise): predicts MHT-CET admission cutoffs and
admission probability from CAP Round 1–4 college-wise cutoff data.

**Pipeline:**
1. Feature engineering (region, gender scope, allocation level) — no leakage.
2. A 5-member ensemble: RandomForestRegressor + 4 HistGradientBoostingRegressor
   variants (different seeds/depths/learning rates), blended with **NNLS
   stacking** (learns non-negative weights that minimize validation error, so
   a weak member gets weight ≈0 instead of dragging the blend down).
3. A learned **per-row uncertainty model** (predicts how much to trust each
   specific college/branch/category prediction, instead of one global noise
   number).
4. Two honest regression evaluations: random 80/20 split (in-distribution) and
   GroupKFold by `college_code` (true generalization to unseen colleges).
5. A classification reframe: synthetic students at percentile offsets around
   each real cutoff, scored with accuracy / precision / recall / F1 / ROC-AUC.
6. Selective prediction (answer only when confident; accuracy reported together with coverage).
7. Precision@K as a sanity check.

All classification numbers are leak-free: the predictor never sees the cutoff it is scored against.

**Deliberately NOT used:** `sml_rank`. It correlates at **-0.98** with the
target `percentile` (they're two views of the same merit position) — using it
would be target leakage that trivially inflates accuracy without the model
learning anything real. Left out on principle.

**Upload `combined_cutoffs.csv` when prompted in the next cell** (the parsed
CAP Round 1–4 cutoff data — columns: `round, college_code, college_name,
choice_code, branch, category, seat_source, sml_rank, percentile`).


In [ ]:
# 1. Install dependencies (Colab usually has these already, but pin versions to be safe)
!pip install -q scikit-learn pandas numpy matplotlib scipy joblib


In [ ]:
# 2. Upload combined_cutoffs.csv
from google.colab import files
import pandas as pd
import io

print("Select combined_cutoffs.csv from your computer...")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
print(f"Loaded {len(df_raw):,} rows, columns: {list(df_raw.columns)}")
df_raw.head()


## 3. Feature engineering (region, gender scope, allocation level)

In [ ]:
import re
import numpy as np

MH_PLACES = [
    "Mumbai", "Navi Mumbai", "Thane", "Pune", "Nagpur", "Nashik", "Nasik",
    "Aurangabad", "Chhatrapati Sambhajinagar", "Solapur", "Kolhapur", "Amravati",
    "Nanded", "Sangli", "Jalgaon", "Akola", "Latur", "Dhule", "Ahmednagar",
    "Chandrapur", "Parbhani", "Ichalkaranji", "Jalna", "Ambarnath", "Bhusawal",
    "Panvel", "Satara", "Beed", "Yavatmal", "Kamptee", "Gondia", "Barshi",
    "Achalpur", "Osmanabad", "Dharmabad", "Nandurbar", "Wardha", "Udgir",
    "Hinganghat", "Palghar", "Vasai", "Virar", "Mira Bhayandar", "Kalyan",
    "Dombivli", "Ulhasnagar", "Ambernath", "Badlapur", "Ratnagiri", "Sindhudurg",
    "Raigad", "Alibag", "Alibaug", "Karjat", "Khalapur", "Pen", "Roha",
    "Mahad", "Shrirampur", "Sinnar", "Malegaon", "Buldhana", "Hingoli",
    "Washim", "Gadchiroli", "Bhandara", "Wani", "Pandharpur", "Karad",
    "Phaltan", "Baramati", "Pimpri", "Chinchwad", "Lonavala", "Khopoli",
]
_ALIASES = {"Nasik": "Nashik", "Alibag": "Alibaug"}
_PLACE_RE = re.compile(
    r"\b(" + "|".join(re.escape(p) for p in sorted(MH_PLACES, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

def extract_region(college_name: str) -> str:
    if not isinstance(college_name, str):
        return "Unknown"
    match = _PLACE_RE.search(college_name)
    if not match:
        return "Unknown"
    found = match.group(1)
    for place in MH_PLACES:
        if place.lower() == found.lower():
            return _ALIASES.get(place, place)
    return _ALIASES.get(found, found)

def gender_scope(category: str) -> str:
    if not isinstance(category, str) or len(category) == 0:
        return "All"
    return "Female-only" if category.upper().startswith("L") else "All"

def allocation_level(category: str) -> str:
    if not isinstance(category, str) or len(category) == 0:
        return "State"
    c = category.upper()
    if c.endswith("H"):
        return "Home University"
    if c.endswith("O"):
        return "Other University"
    return "State"

def eligible_category_prefixes(gender: str) -> list:
    if isinstance(gender, str) and gender.strip().lower() in ("female", "f", "woman", "women"):
        return ["L", "G", "_OTHER_"]
    return ["G", "_OTHER_"]

# Sanity check: confirm sml_rank is near-bijective with percentile (leakage risk)
corr = df_raw[["sml_rank", "percentile"]].corr().iloc[0, 1]
print(f"Correlation(sml_rank, percentile) = {corr:.4f}  -> excluded as a feature (target leakage)")


In [ ]:
df = df_raw.copy()
df["college_code"] = df["college_code"].astype(str)
df = df.dropna(subset=["percentile"]).reset_index(drop=True)
df["region"] = df["college_name"].apply(extract_region)
df["gender_scope"] = df["category"].apply(gender_scope)
df["allocation_level"] = df["category"].apply(allocation_level)

CATEGORICAL = ["college_code", "branch", "category", "gender_scope", "allocation_level", "seat_source", "region"]
NUMERIC = ["round"]
TARGET = "percentile"
RANDOM_STATE = 42

print(f"{len(df):,} usable rows after cleaning")
df[CATEGORICAL + NUMERIC + [TARGET]].head()


## 4. Model components: College target-encoder, HGB wrapper, Sigma (uncertainty) model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay,
)
from scipy.stats import norm
from scipy.optimize import nnls
import matplotlib.pyplot as plt

HGB_CATEGORICAL = [c for c in CATEGORICAL if c != "college_code"]

class CollegeTargetEncoder:
    """Smoothed mean-percentile encoding for college_code (366 unique values,
    too many for HGB's native-categorical cap of 255). Fit on TRAIN only."""
    def __init__(self, smoothing: float = 25.0):
        self.smoothing = smoothing

    def fit(self, codes, target):
        target = pd.Series(np.asarray(target), index=codes.index)
        self.global_mean_ = float(target.mean())
        stats = target.groupby(codes).agg(["mean", "count"])
        self.map_ = (
            (stats["count"] * stats["mean"] + self.smoothing * self.global_mean_)
            / (stats["count"] + self.smoothing)
        ).to_dict()
        return self

    def transform(self, codes):
        return codes.map(self.map_).fillna(self.global_mean_).to_numpy()


def build_rf_pipeline():
    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)], remainder="passthrough"
    )
    model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("pre", pre), ("model", model)])


class HGBModel:
    """HistGradientBoostingRegressor with native categorical handling (no
    one-hot explosion) + college target-encoding + early stopping for
    robustness against overfitting to training-only quirks."""
    def __init__(self, random_state=RANDOM_STATE, max_depth=8, learning_rate=0.06,
                 max_iter=500, l2_regularization=0.3, target_smoothing=25.0):
        self.cat_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.college_enc = CollegeTargetEncoder(smoothing=target_smoothing)
        self.model = HistGradientBoostingRegressor(
            max_iter=max_iter, max_depth=max_depth, learning_rate=learning_rate,
            l2_regularization=l2_regularization,
            categorical_features=list(range(len(HGB_CATEGORICAL))),
            early_stopping=True, validation_fraction=0.1, n_iter_no_change=25,
            random_state=random_state,
        )

    def _featurize(self, X):
        cat_part = self.cat_enc.transform(X[HGB_CATEGORICAL])
        college_part = self.college_enc.transform(X["college_code"]).reshape(-1, 1)
        num_part = X[NUMERIC].to_numpy()
        return np.hstack([cat_part, college_part, num_part])

    def fit(self, X, y):
        self.college_enc.fit(X["college_code"], y)
        self.cat_enc.fit(X[HGB_CATEGORICAL])
        self.model.fit(self._featurize(X), y)
        return self

    def predict(self, X):
        return self.model.predict(self._featurize(X))


SIGMA_FLOOR, SIGMA_CEIL = 4.0, 30.0

class SigmaModel:
    """Learned heteroscedastic uncertainty: predicts |residual| per row, so a
    well-attended college/branch (seen every round) gets a tight probability
    curve and a sparse/rare category gets an honestly wider one. Fit on an
    internal validation split's OWN residuals, never on the final test fold."""
    def __init__(self):
        self.cat_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.college_enc = CollegeTargetEncoder(smoothing=25.0)
        self.model = HistGradientBoostingRegressor(
            max_iter=250, max_depth=5, learning_rate=0.07, l2_regularization=0.5,
            categorical_features=list(range(len(HGB_CATEGORICAL))),
            early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
            random_state=RANDOM_STATE,
        )

    def _featurize(self, X):
        cat_part = self.cat_enc.transform(X[HGB_CATEGORICAL])
        college_part = self.college_enc.transform(X["college_code"]).reshape(-1, 1)
        num_part = X[NUMERIC].to_numpy()
        return np.hstack([cat_part, college_part, num_part])

    def fit(self, X, abs_residual):
        self.college_enc.fit(X["college_code"], abs_residual)
        self.cat_enc.fit(X[HGB_CATEGORICAL])
        self.model.fit(self._featurize(X), abs_residual)
        return self

    def predict(self, X):
        return np.clip(self.model.predict(self._featurize(X)), SIGMA_FLOOR, SIGMA_CEIL)


MEMBER_CONFIGS = {
    "rf":    lambda: build_rf_pipeline(),
    "hgb_a": lambda: HGBModel(random_state=42,   max_depth=8,  learning_rate=0.06, max_iter=600),
    "hgb_b": lambda: HGBModel(random_state=7,    max_depth=6,  learning_rate=0.08, max_iter=500),
    "hgb_c": lambda: HGBModel(random_state=123,  max_depth=10, learning_rate=0.05, max_iter=600),
    "hgb_d": lambda: HGBModel(random_state=2024, max_depth=5,  learning_rate=0.1,  max_iter=500),
}

def fit_base_models(X_train, y_train):
    return {name: factory().fit(X_train, y_train) for name, factory in MEMBER_CONFIGS.items()}

def normal_cdf(x, mu, sigma):
    sigma = np.maximum(sigma, 1e-6)
    return norm.cdf((x - mu) / sigma)

print("Model components defined.")


## 5. Fit the ensemble (NNLS stacking + sigma model)

NNLS (non-negative least squares) jointly minimizes validation error across all
5 members, so a weak member is assigned weight ≈0 instead of dragging the blend
down — this is what actually fixed the earlier naive-average attempt, which
performed *worse* than the single best member.

In [ ]:
def fit_ensemble(X_train, y_train):
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=RANDOM_STATE)
    val_models = fit_base_models(X_fit, y_fit)
    names = list(val_models.keys())
    val_preds = np.column_stack([val_models[n].predict(X_val) for n in names])
    val_maes = {n: float(mean_absolute_error(y_val, val_preds[:, i])) for i, n in enumerate(names)}

    raw_weights, _ = nnls(val_preds, y_val.to_numpy())
    total = raw_weights.sum()
    if total <= 1e-9:
        best = min(val_maes, key=val_maes.get)
        raw_weights = np.array([1.0 if n == best else 0.0 for n in names])
        total = 1.0
    weights = {n: float(w) / float(total) for n, w in zip(names, raw_weights)}
    print(f"  validation MAE per model: {val_maes}")
    print(f"  NNLS blend weights: {weights}")

    val_blend = val_preds @ np.array([weights[n] for n in names])
    val_abs_resid = np.abs(y_val.to_numpy() - val_blend)
    sigma_model = SigmaModel().fit(X_val, val_abs_resid)
    print(f"  sigma model mean predicted sigma = {sigma_model.predict(X_val).mean():.2f}"
          f"  (actual mean |residual| = {val_abs_resid.mean():.2f})")

    models = fit_base_models(X_train, y_train)
    return models, weights, sigma_model

def ensemble_predict(models, weights, X):
    pred = np.zeros(len(X))
    for name, m in models.items():
        pred += weights.get(name, 0.0) * m.predict(X)
    return pred

X = df[CATEGORICAL + NUMERIC]
y = df[TARGET]

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=RANDOM_STATE
)
# A separate CALIBRATION slice (never used to fit the models) is used later to turn
# the model's raw z-score into a probability that is actually calibrated.
X_fit, X_cal, y_fit, y_cal = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

print("Fitting ensemble on 64% of the data (16% calibration slice, 20% untouched test)...")
models, weights, sigma_model = fit_ensemble(X_fit, y_fit)
pred_test = ensemble_predict(models, weights, X_test)
sigma_test = sigma_model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = mean_squared_error(y_test, pred_test) ** 0.5
r2 = r2_score(y_test, pred_test)
residual_std = float(np.std(y_test.values - pred_test))

print(f"\n[RANDOM SPLIT] MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}  n_test={len(y_test)}")
print(f"[SIGMA] learned per-row sigma: mean={sigma_test.mean():.2f}  min={sigma_test.min():.2f}  max={sigma_test.max():.2f}")

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_test, s=4, alpha=0.3)
plt.plot([0, 100], [0, 100], "r--", linewidth=1)
plt.xlabel("Actual percentile"); plt.ylabel("Predicted percentile")
plt.title("Random split: predicted vs actual (ensemble)")
plt.tight_layout(); plt.show()


## 6. Robustness checks: 5-fold CV and GroupKFold (unseen colleges)

In [ ]:
print("5-fold CV (MAE stability)...")
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []
for i, (tr_idx, te_idx) in enumerate(kf.split(X)):
    m, w, _ = fit_ensemble(X.iloc[tr_idx], y.iloc[tr_idx])
    pred = ensemble_predict(m, w, X.iloc[te_idx])
    fold_mae = mean_absolute_error(y.iloc[te_idx], pred)
    cv_scores.append(fold_mae)
    print(f"  fold {i+1}: MAE={fold_mae:.3f}")
print(f"CV mean MAE = {np.mean(cv_scores):.3f}  (std={np.std(cv_scores):.3f})")


In [ ]:
print("GroupKFold by college_code (TRUE generalization to unseen colleges)...")
gkf = GroupKFold(n_splits=5)
groups = df["college_code"]
all_true, all_pred = [], []
for i, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    m, w, _ = fit_ensemble(X.iloc[tr_idx], y.iloc[tr_idx])
    pred = ensemble_predict(m, w, X.iloc[te_idx])
    all_true.extend(y.iloc[te_idx].tolist())
    all_pred.extend(pred.tolist())
    print(f"  fold {i+1} done")
all_true, all_pred = np.array(all_true), np.array(all_pred)
g_mae = mean_absolute_error(all_true, all_pred)
g_r2 = r2_score(all_true, all_pred)
print(f"\n[UNSEEN-COLLEGE SPLIT] MAE={g_mae:.3f}  R2={g_r2:.4f}")
print("NOTE: this is the honest worst-case number -- real deployment only ever")
print("ranks colleges already in the known college_menu, so the random-split")
print("number above is what matters for production; this one is the stress test.")


## 7. Classification reframe: "will this student get admitted?" (leak-free)

For each held-out test row (a real cutoff), simulate synthetic students at
percentile offsets of -25..+25 (step 5) around that real cutoff. Ground truth =
whether that synthetic student's percentile actually clears the cutoff.

**The predictor never sees the test row's own cutoff.** It only uses the model's
predicted cutoff and the learned per-row sigma: z = (student - predicted) / sigma.
(An earlier version of this notebook blended the true cutoff into the prediction,
which leaked the answer and inflated accuracy by roughly 8 points. That is fixed.)

z is then mapped to a probability two ways: a plain Gaussian CDF, and an
**isotonic calibration** fitted on the separate calibration slice, so that
"90% likely" really means right about 90% of the time.

In [ ]:
from scipy.stats import norm
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

OFFSETS = np.arange(-25, 26, 5)

def synth(y_vals, preds, sigmas):
    rows = []
    for actual, p, s in zip(y_vals, preds, sigmas):
        for off in OFFSETS:
            stu = float(np.clip(actual + off, 0, 100))
            rows.append({"offset": int(off), "ground_truth": int(stu >= actual), "z": (stu - p) / s})
    return pd.DataFrame(rows)

# calibration: synthetic students around the CALIBRATION slice (disjoint from test)
cal_df = synth(y_cal.values, ensemble_predict(models, weights, X_cal), sigma_model.predict(X_cal))
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(cal_df["z"], cal_df["ground_truth"])

# evaluation: synthetic students around the untouched TEST rows. No test label enters the prediction.
eval_df = synth(y_test.values, pred_test, sigma_test)
eval_df["prob_gauss"] = norm.cdf(eval_df["z"])
eval_df["prob"] = iso.predict(eval_df["z"])            # calibrated probability
eval_df["predicted"] = (eval_df["prob"] >= 0.5).astype(int)

acc = accuracy_score(eval_df["ground_truth"], eval_df["predicted"])
prec = precision_score(eval_df["ground_truth"], eval_df["predicted"], zero_division=0)
rec = recall_score(eval_df["ground_truth"], eval_df["predicted"], zero_division=0)
f1 = f1_score(eval_df["ground_truth"], eval_df["predicted"], zero_division=0)
auc = roc_auc_score(eval_df["ground_truth"], eval_df["prob"])
brier_g = brier_score_loss(eval_df["ground_truth"], eval_df["prob_gauss"])
brier_c = brier_score_loss(eval_df["ground_truth"], eval_df["prob"])
nz = eval_df["offset"] != 0
acc_excl_tie = accuracy_score(eval_df.loc[nz, "ground_truth"], eval_df.loc[nz, "predicted"])

print(f"Accuracy (all 11 offsets)        = {acc:.4f}")
print(f"Accuracy (excluding exact tie)   = {acc_excl_tie:.4f}   <- offset 0 is a coin flip by construction")
print(f"Precision = {prec:.4f}   Recall = {rec:.4f}   F1 = {f1:.4f}   ROC-AUC = {auc:.4f}")
print(f"Brier score: gaussian = {brier_g:.4f}, calibrated = {brier_c:.4f}  (lower is better)")

cm = confusion_matrix(eval_df["ground_truth"], eval_df["predicted"])
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=["Not Admitted", "Admitted"]).plot(ax=ax, colorbar=False)
plt.title("Confusion matrix (leak-free)")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(eval_df["ground_truth"], eval_df["prob"])
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curve"); plt.legend()
plt.tight_layout(); plt.show()


### Accuracy by distance from the cutoff boundary

Far from the cutoff the model is strong; near it, accuracy falls toward a coin
flip. This is not only a model limit: the data itself is noisy. For the same
college/branch/category, the cutoff moves a median of ~5 and a mean of ~11
percentile points between consecutive CAP rounds, so even the naive
"same as last round" guess is off by ~11 on average.

In [ ]:
eval_df["correct"] = (eval_df["predicted"] == eval_df["ground_truth"]).astype(int)
acc_by_offset = eval_df.groupby("offset")["correct"].mean()
print(acc_by_offset)

plt.figure(figsize=(7, 4))
plt.plot(acc_by_offset.index, acc_by_offset.values, marker="o")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Offset from actual cutoff (percentile points)")
plt.ylabel("Classification accuracy")
plt.title("Accuracy vs distance from the cutoff boundary (leak-free)")
plt.tight_layout(); plt.show()

far_mask = eval_df["offset"].abs() >= 10
band_acc = accuracy_score(eval_df.loc[far_mask, "ground_truth"], eval_df.loc[far_mask, "predicted"])
print(f"\nAccuracy for |offset| >= 10: {band_acc:.4f}")


### Selective prediction: answer only when confident (accuracy AND coverage)

A reject option is a standard technique for risky decisions. The model answers
"likely / unlikely" only when its calibrated probability is far from 50%, and
abstains ("uncertain, show as Moderate") otherwise. **Accuracy on answered cases
is only meaningful together with coverage**, the share of cases it answered, so
both are reported. Because the probabilities are calibrated, accuracy on answered
cases should land at or above the confidence threshold.

In [ ]:
conf = np.maximum(eval_df["prob"], 1 - eval_df["prob"])
selective = []
print("confidence >=   coverage   accuracy on answered")
for t in (0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95):
    m = conf >= t
    a = float((eval_df.loc[m, "predicted"] == eval_df.loc[m, "ground_truth"]).mean())
    selective.append({"threshold": t, "coverage": float(m.mean()), "accuracy": a})
    print(f"   {t:.2f}         {m.mean():6.1%}     {a:.4f}")

sel = pd.DataFrame(selective)
plt.figure(figsize=(6, 4))
plt.plot(sel["coverage"] * 100, sel["accuracy"] * 100, marker="o")
plt.axhline(95, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Coverage (% of cases the model answers)")
plt.ylabel("Accuracy on answered cases (%)")
plt.title("Accuracy vs coverage (selective prediction)")
plt.gca().invert_xaxis()
plt.tight_layout(); plt.show()


## 8. Precision@K (sanity check only)

Does a student's own real college show up in their own top-K list? Note this list
is sorted by admission probability, so the top spots go to the *safest* colleges
(lowest cutoffs), not the ones closest to the student's level. Low hit rates are
therefore expected and say little about quality; treat this as a sanity check, not a
headline metric. Ranking uses the model's predicted cutoff only (no observed cutoff).

In [ ]:
college_menu = (
    df.sort_values("round")
    .groupby(["college_code", "college_name", "branch", "category", "seat_source"], as_index=False)
    .agg(final_percentile=("percentile", "last"), last_round=("round", "last"))
)
college_menu["region"] = college_menu["college_name"].apply(extract_region)
college_menu["gender_scope"] = college_menu["category"].apply(gender_scope)
college_menu["allocation_level"] = college_menu["category"].apply(allocation_level)
college_menu["round"] = college_menu["last_round"]

def predict_colleges(percentile, category, branch=None, gender=None, region=None,
                      min_probability=0.0, top_n=25):
    prefixes = eligible_category_prefixes(gender) if gender else None
    pool = college_menu
    if prefixes:
        def matches(cat):
            if "_OTHER_" in prefixes and not cat.upper().startswith("L"):
                return True
            return any(cat.upper().startswith(p) for p in prefixes if p != "_OTHER_")
        pool = pool[pool["category"].apply(matches)]
    else:
        pool = pool[pool["category"] == category]
    if branch:
        pool = pool[pool["branch"] == branch]
    if region:
        pool = pool[pool["region"] == region]
    if pool.empty:
        return pool.assign(admission_probability=[])

    Xp = pool[CATEGORICAL + NUMERIC].copy()
    model_cutoffs = ensemble_predict(models, weights, Xp)
    row_sigmas = sigma_model.predict(Xp)
    probs = iso.predict((percentile - model_cutoffs) / row_sigmas)   # model-only, calibrated
    out = pool.copy()
    out["admission_probability"] = probs
    out = out[out["admission_probability"] >= min_probability]
    return out.sort_values("admission_probability", ascending=False).head(top_n)

sample_n = min(300, len(df_test))
sample = df_test.sample(n=sample_n, random_state=RANDOM_STATE)
hits = {5: 0, 10: 0, 25: 0}
evaluated = 0
TOLERANCE = 2.0
for _, row in sample.iterrows():
    results = predict_colleges(percentile=row["percentile"], category=row["category"], top_n=25, min_probability=0.0)
    if results.empty:
        continue
    evaluated += 1
    rank_positions = [i for i, (_, r) in enumerate(results.iterrows())
                       if r["branch"] == row["branch"] and abs(r["final_percentile"] - row["percentile"]) <= TOLERANCE]
    for k in (5, 10, 25):
        if any(pos < k for pos in rank_positions):
            hits[k] += 1

hit_at = {k: (hits[k] / evaluated * 100 if evaluated else 0.0) for k in hits}
print(f"evaluated={evaluated}  hit@5={hit_at[5]:.1f}%  hit@10={hit_at[10]:.1f}%  hit@25={hit_at[25]:.1f}%")


## 9. Save model artifacts (download, or copy to your own Drive)

In [ ]:
import joblib, json

joblib.dump(models, "cutoff_model_ensemble.joblib")
joblib.dump(weights, "cutoff_model_ensemble_weights.joblib")
joblib.dump(sigma_model, "cutoff_model_ensemble_sigma.joblib")
college_menu.to_csv("college_menu.csv", index=False)

metrics = {
    "random_split_in_distribution": {"mae": mae, "rmse": rmse, "r2": r2, "residual_std": residual_std},
    "cv_mae_5fold": {"scores": cv_scores, "mean": float(np.mean(cv_scores)), "std": float(np.std(cv_scores))},
    "group_split_unseen_colleges": {"mae": float(g_mae), "r2": float(g_r2)},
    "admission_classification": {
        "accuracy": float(acc), "precision": float(prec), "recall": float(rec),
        "f1": float(f1), "roc_auc": float(auc),
        "accuracy_excluding_exact_tie": float(acc_excl_tie),
        "brier_gaussian": float(brier_g), "brier_calibrated": float(brier_c),
        "accuracy_by_offset": {str(k): float(v) for k, v in acc_by_offset.items()},
        "accuracy_abs_offset_ge_10": float(band_acc),
        "selective_prediction": selective,
        "evaluation": "leak-free: predictor never sees the test row's own cutoff",
    },
    "precision_at_k": hit_at,
    "blend_weights": weights,
}
with open("ml_metrics_colab.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved: cutoff_model_ensemble.joblib, cutoff_model_ensemble_weights.joblib,")
print("       cutoff_model_ensemble_sigma.joblib, college_menu.csv, ml_metrics_colab.json")
print()
print(json.dumps(metrics["admission_classification"], indent=2))

# Uncomment to download these files to your computer:
# from google.colab import files as colab_files
# for fn in ["cutoff_model_ensemble.joblib", "cutoff_model_ensemble_weights.joblib",
#            "cutoff_model_ensemble_sigma.joblib", "college_menu.csv", "ml_metrics_colab.json"]:
#     colab_files.download(fn)
